# 02 · Demand EDA (Team 10, Deliverable 3)

**Question:** what does passenger demand look like over time and across routes, and what does
that say about how to forecast it two years ahead?

Each section ends with a decision. Where a decision rests on a threshold (the disruption
window, the growth cap), the notebook prints a **CHECK** line, so a run on the real data shows
at a glance whether the assumption still holds.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODELING = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "team-10-modeling"]
                if (p / "src" / "forecasting.py").exists())
sys.path.insert(0, str(MODELING / "src"))

from data import BANNER, FIGS, load_airports, load_pax, tag
from features import DISRUPTION, GROWTH_CAP, KEY, airport_table, clean_pax, route_table, shift_months

print(BANNER)
pax = clean_pax(load_pax())
routes = route_table(pax, airport_table(load_airports()))


def check(label: str, ok: bool, detail: str = "") -> None:
    print(f"CHECK [{'PASS' if ok else 'REVIEW'}] {label} {detail}")

## 1. The target is heavily skewed

In [ ]:
y = pax["OD_PAX"]
print(f"skew {y.skew():.1f} | skew of log1p {np.log1p(y).skew():.2f} | mean/median {y.mean() / y.median():.1f}")
vol = pax.groupby(KEY)["OD_PAX"].sum().sort_values(ascending=False)
top10 = vol.head(max(1, len(vol) // 10)).sum() / vol.sum()
print(f"top 10% of routes carry {top10:.0%} of all passengers")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(y), bins=60)
ax.set(xlabel="log10(OD_PAX)", ylabel="route-months", title=tag("Route-month passengers (log scale)"))
fig.tight_layout(); fig.savefig(FIGS / "eda_target_distribution.png", dpi=120); plt.show()

**Therefore:** score with **WAPE** (total absolute error ÷ total passengers), which weights
routes by size the way a network plan does, and report it by route-size segment too, so small
routes are not hidden. Fit growth on a **log** scale.

## 2. The time axis: trend, seasonality, disruption

In [ ]:
net = pax.groupby("DATE")["OD_PAX"].sum()
fig, ax = plt.subplots(figsize=(12, 4))
net.plot(ax=ax, color="black")
ax.axvspan(DISRUPTION[0], DISRUPTION[1], color="tab:red", alpha=0.12, label="disruption window (excluded from training pairs)")
ax.set(ylabel="passengers", title=tag("Network passengers per month"))
ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "eda_network_series.png", dpi=120); plt.show()

yoy = (net / net.shift(12) - 1).dropna()
inside = yoy[(yoy.index >= DISRUPTION[0]) & (yoy.index <= DISRUPTION[1] + pd.DateOffset(months=12))]
outside = yoy.drop(inside.index)
print(f"network YoY change, months outside disruption (+12m echo): median {outside.median():+.1%}, "
      f"range {outside.min():+.1%} .. {outside.max():+.1%}")
check("network YoY outside the window stays within the growth cap",
      bool(((1 + outside) >= GROWTH_CAP[0]).all() and ((1 + outside) <= GROWTH_CAP[1]).all()),
      f"(cap {GROWTH_CAP})")
first_normal = yoy[(yoy.index > DISRUPTION[1] + pd.DateOffset(months=12))]
print("months whose month AND anchor are both after the window, with YoY still outside the cap:",
      ", ".join(d.strftime("%Y-%m") for d in first_normal[(1 + first_normal < GROWTH_CAP[0]) | (1 + first_normal > GROWTH_CAP[1])].index) or "none")

The year-over-year change of a month *and* of its anchor a year earlier must both be normal for
a training pair to teach normal growth. Hence training pairs are dropped when either month falls
in the window. If the list above names any month, the window should be extended
(`DISRUPTION` in `src/features.py`) before D4 is run on the real data.

In [ ]:
d = pax.merge(routes[[*KEY, "is_international"]], on=KEY, how="left")
d = d[~((d["DATE"] >= DISRUPTION[0]) & (d["DATE"] <= DISRUPTION[1]))]
d["market"] = np.where(d["is_international"] == 1, "international", "domestic")
prof = d.groupby(["market", d["DATE"].dt.month])["OD_PAX"].sum().unstack(0)
prof = prof / prof.mean()
fig, ax = plt.subplots(figsize=(8, 4))
prof.plot(ax=ax, marker="o")
ax.set(xlabel="month", ylabel="relative to market's average month",
       title=tag("Seasonal shape outside the disruption window"))
fig.tight_layout(); fig.savefig(FIGS / "eda_seasonality.png", dpi=120); plt.show()
print("peak-to-trough ratio:", (prof.max() / prof.min()).round(2).to_dict())

**Therefore:**

- There **is** a genuine time axis this time (unlike the Kaggle data), so validation is by
  **forecast origin**: train on months before T, forecast T..T+23 once.
- The yearly cycle is strong and differs by market. So the reference forecast is the
  **seasonal naive** (same route, same month, year before T), and any model has to beat it.
- Network-level ARIMA would forecast one series and cannot say which route the passengers
  are on, so it is not a candidate for a route plan.

## 3. How much do routes grow from one year to the next?

In [ ]:
pairs = pax.assign(anchor_date=shift_months(pax["DATE"], -12)).merge(
    pax.rename(columns={"DATE": "anchor_date", "OD_PAX": "anchor"}), on=[*KEY, "anchor_date"])
clean = pairs[~((pairs["DATE"] >= DISRUPTION[0]) & (pairs["DATE"] <= DISRUPTION[1]))
              & ~((pairs["anchor_date"] >= DISRUPTION[0]) & (pairs["anchor_date"] <= DISRUPTION[1]))]
annual = clean.groupby([*KEY, clean["DATE"].dt.year]).agg(pax=("OD_PAX", "sum"), anchor=("anchor", "sum"), n=("OD_PAX", "size"))
annual = annual[annual["n"] >= 6]
g = annual["pax"] / annual["anchor"]
w = annual["pax"]
within = float(w[(g >= GROWTH_CAP[0]) & (g <= GROWTH_CAP[1])].sum() / w.sum())
print(f"route-years (>= 6 paired months): {len(g):,}; median growth {g.median():.3f}")
print(f"share of passengers on route-years with growth inside the cap: {within:.1%}")
check("growth cap covers >= 90% of passengers on clean route-years", within >= 0.9)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log(g.clip(0.2, 5)), bins=60)
for c in GROWTH_CAP:
    ax.axvline(np.log(c), color="tab:red", ls="--")
ax.set(xlabel="log annual growth (route-year)", ylabel="route-years", title=tag("Year-over-year route growth, clean pairs"))
fig.tight_layout(); fig.savefig(FIGS / "eda_route_growth.png", dpi=120); plt.show()

**Therefore:** fitted models predict **log growth on top of the seasonal naive**, capped at
`GROWTH_CAP` per year, rather than raw passenger levels. A level model trained across 2019–2022
learns the collapse and the rebound as if they were a trend and extrapolates them. A growth model
only has to explain the spread in the histogram above.

## 4. Model-family view

| EDA observation | Points toward / away from |
|---|---|
| Strong, market-specific yearly cycle | Toward a seasonal reference (seasonal naive) that every model must beat |
| Real time axis with a 2020 break | Toward origin-based validation and training pairs that skip the break; away from random CV |
| Panel of many routes, very skewed sizes | Toward route-level forecasts scored by WAPE, with size segments; away from one network series (ARIMA) |
| Year-over-year growth mostly within a narrow band | Toward modelling **growth** (Ridge or gradient-boosted trees on route/airport/calendar inputs) instead of levels |
| Only ~5–7 clean years | Away from LSTM (short history, new dependency) |

**Named family for D4:** *seasonal naive as the reference, with year-over-year growth models
(Ridge, gradient-boosted trees) as the challengers.*